# Compare logical state-preparation circuits

Fault-tolerant quantum computations consume code blocks prepared in logical states such as $|0_L\rangle$ and $|+_L\rangle$.
The errors that a noisy preparation circuit leaves on a block matter through their effect on the circuit that uses the block next, which we call its consumer.
A logical error rate of the prepared state misses some of these errors.
For $|0_L\rangle$, it checks only $Z_L$, which no Z-type error flips.
It also does not count errors that a decoder corrects, although such errors can make a consumer fail in combination with the consumer's own noise.
This notebook takes a short run, or window, of Steane error-correction steps as the consumer, calibrates it once, and reuses the calibration to compare preparation circuits.

The preparation circuits post-select on flag measurements.
The resulting figure of merit, $\Phi$, is conditioned on acceptance: it estimates the signed change in the window's failure probability, per accepted preparation, relative to an ideal block.
We therefore report it together with the acceptance probability.
The [consumer-calibration guide](https://github.com/qLDPCOrg/qLDPC/blob/main/src/qldpc/experimental/circuits/benchmarking/consumer_calibration.md) defines $\Phi$ and explains its approximations.
These tools come from `qldpc.experimental`, so their interface may still change.

## Setup

Run the next cell once in a fresh notebook environment.
IPython's `%pip` magic installs into the active kernel, so the following imports use the same environment.

In [1]:
%%capture
%pip install qldpc

In [2]:
import math

import stim

from qldpc import circuits, codes
from qldpc.experimental.circuits import (
    ConsumerProfile,
    estimate_downstream_fault_weight,
    estimate_residual_weight_histogram,
    get_steane_swap_consumer,
)

## A preparation circuit to benchmark

We benchmark the flagged preparation of the Steane code's $|0_L\rangle$ state from Figure 1c of [Goto (2016)](https://www.nature.com/articles/srep19578), under circuit-level depolarizing noise at physical error rate $p = 10^{-3}$.
Its flag qubit, qubit 7, detects single circuit errors that would otherwise leave a heavy error on the block.
A repeat-until-success protocol prepares the block again whenever the flag fires.
The option `add_flags=True` models this protocol: it adds a detector for the flag measurement and, by default, discards the shots in which it fires.
By default, the benchmark takes the prepared block to be the qubits that the circuit acts on but never measures, here qubits 0 through 6.
Stim describes the noise of a circuit as a list of error mechanisms, independent random events such as a Pauli error after a gate.
A fault is the set of error mechanisms that occur in one shot, and its weight is their number.
The Steane code corrects every error of weight at most $t = 1$.

In [3]:
code = codes.SteaneCode()
prep_circuit = stim.Circuit("""
    # non-fault-tolerant preparation of the logical zero state
    H 0 2 4
    CX 0 3 2 1 4 5
    CX 0 1 2 6 4 3
    CX 2 5 3 6
    # flag qubit 7 detects errors in a logical Z representative
    H 7
    CZ 7 1 7 3 7 5
    MX 7
""")
error_rate = 1e-3
noise_model = circuits.DepolarizingNoiseModel(error_rate)
noisy_circuit = noise_model.noisy_circuit(prep_circuit)
options = {"add_flags": True}

## Calibrate once, score many preparations

`get_steane_swap_consumer(code, noise_model)` builds a consumer from swap-type Steane error correction, as in the [Steane-QEC notebook](../logical_error_rates/6_steane_qec.ipynb).
Swap-type Steane error correction teleports the logical qubit from block to block, by steps of two alternating kinds.
A Z-correction step applies a transversal CNOT from the current block to a fresh $|0_L\rangle$ block, and measures the current block in the X basis to detect Z errors.
An X-correction step applies a transversal CNOT from a fresh $|+_L\rangle$ block to the current block, and measures the current block in the Z basis to detect X errors.

The block under test is a fresh $|0_L\rangle$ block.
Three steps decode its errors: the step by which it enters and the next two.
They decode its Z errors twice and its X errors once.
Faults in the two steps before entry also reach these decodings, so the consumer runs a window of five noisy steps, with the block under test entering by the third.
The noise model acts on the CNOTs and measurements of the steps, and the data block and the other fresh blocks are ideal.
By default, a lookup-table decoder decodes the syndrome of each step on its own.
The calibration therefore measures the cost of one imperfect fresh $|0_L\rangle$ block.
The [consumer-calibration guide](https://github.com/qLDPCOrg/qLDPC/blob/main/src/qldpc/experimental/circuits/benchmarking/consumer_calibration.md#the-window) traces how each type of error travels through the window.

The residual weight of the X-type or Z-type part of a block's error is its smallest weight up to stabilizers of the prepared state.
The `calibrate` method estimates how much the window's failure probability rises for each Pauli type and residual weight.
The calibration's `estimate_excess_failure_probability` method then scores a preparation circuit from its residual-weight histograms, the distributions of these weights over its accepted shots, without simulating the window again.
It also checks that the circuit prepares the calibrated state.
The seeds make the outputs reproducible.

In [4]:
consumer = get_steane_swap_consumer(code, noise_model)
calibration = consumer.calibrate(seed=0)
phi, standard_error = calibration.estimate_excess_failure_probability(
    noisy_circuit, seed=0, **options
)
print(f"Phi = {phi:.2e} (standard error {standard_error:.1e})")

Phi = 1.92e-04 (standard error 1.1e-06)


The calibrated result is a signed change of $1.92\times10^{-4}$ in the window's failure probability per accepted preparation, relative to an ideal block.
The standard error covers sampling in the calibration and in the residual-weight histograms.
It does not cover the approximations of $\Phi$ that the consumer-calibration guide describes.
One of them adds the X-type and Z-type contributions separately.
Another uses, for each residual weight, a cost averaged over sampled errors rather than over the errors that this circuit leaves.
A direct simulation of the window on the errors that the circuit actually leaves avoids these approximations, though it shares the window's own.
It must be repeated for every circuit, however.
In the next cell, 20 million shots of direct simulation still give a standard error about three times that of $\Phi$.

In [5]:
direct, direct_error = consumer.estimate_excess_failure_probability(
    noisy_circuit, 20_000_000, seed=0, **options
)
difference = (phi - direct) / math.hypot(standard_error, direct_error)
print(f"direct simulation: {direct:.2e} (standard error {direct_error:.1e})")
print(f"Phi - direct: {difference:.1f} combined standard errors")

direct simulation: 1.88e-04 (standard error 3.1e-06)
Phi - direct: 1.2 combined standard errors


The direct estimate is about 2% below $\Phi$, a difference of 1.2 combined standard errors, so the two agree within sampling error.
This check therefore finds no effect of the approximations at this precision.
One of them is that $\Phi$ adds the contributions of the X and Z parts of each error, which neglects their interaction when both occur on the block, as for Y errors or X and Z errors from one fault.
The [validation section](https://github.com/qLDPCOrg/qLDPC/blob/main/src/qldpc/experimental/circuits/benchmarking/consumer_calibration.md#how-to-validate) of the consumer-calibration guide describes a paired direct check that estimates this interaction.

## Phase noise after preparation

The next cell adds a Z error with probability $p$ to each qubit of the block after the preparation, as an idle period might.
This noise leaves $Z_L$ unchanged, so a $Z_L$ readout of $|0_L\rangle$ does not detect it.
The window is unchanged, so the same calibration scores the new circuit.

In [6]:
z_idling = stim.Circuit(f"Z_ERROR({error_rate}) " + " ".join(map(str, range(len(code)))))
idle_phi, _ = calibration.estimate_excess_failure_probability(
    noisy_circuit + z_idling, seed=0, **options
)
print(f"with Z idling: Phi = {idle_phi:.2e} (without: {phi:.2e})")

with Z idling: Phi = 3.78e-04 (without: 1.92e-04)


$\Phi$ nearly doubles.
For the Steane code, every Z-type error on $|0_L\rangle$ is equivalent to one of weight at most one, which the code corrects.
These errors never make the window fail alone, so the whole increase comes from their combination with the window's own faults.
A $Z_L$ readout of the prepared state would not see this increase.

## Comparing preparation protocols

The next cell compares the flagged protocol with the same circuit run without post-selection on its flag.
Because $\Phi$ is conditioned on acceptance, we compare both $\Phi$ and the acceptance probability.
The acceptance probability sets how often a preparation must be repeated.

In [7]:
protocols = {
    "with post-selection": options,
    "without post-selection": {**options, "post_select": False},
}
histograms = {}
for label, protocol_options in protocols.items():
    histograms[label] = estimate_residual_weight_histogram(
        noisy_circuit, max_residual_weight=consumer.correctable_weight, seed=0, **protocol_options
    )
    protocol_phi, _ = calibration.estimate_excess_failure_probability(
        noisy_circuit, seed=0, **protocol_options
    )
    acceptance = histograms[label].acceptance_probability
    print(f"{label}: keeps {acceptance:.4f} of shots, Phi {protocol_phi:.2e}")

with post-selection: keeps 0.9925 of shots, Phi 1.92e-04


without post-selection: keeps 1.0000 of shots, Phi 2.16e-03


Without post-selection, every preparation is kept, but $\Phi$ is more than ten times larger.
The shots that the flag would discard therefore account for about 90% of $\Phi$ without post-selection.
In these shots, a fault of weight one leaves a heavy error, here an X error of residual weight two, more than the Steane code corrects.
The window fails on such an error even without faults of its own.
Pairs of $\Phi$ and acceptance probability compare candidate circuits and post-selection rules on both axes.

## An uncalibrated alternative

`ConsumerProfile.poisson` models a consumer without simulating one.
The modeled consumer fails exactly when $m + g > t$.
Here $m$ is the residual weight of the block's whole error, and $g$ is the weight of the consumer's own fault, which the model takes to be Poisson-distributed.
`estimate_downstream_fault_weight` estimates the mean $\nu$ of $g$ from the noise of the preparation circuit, for a reference consumer: one round of Steane-type error correction.
This reference consumer differs from the window above, whose expected fault weight is the sum of the probabilities of its error mechanisms.
The model costs only a residual-weight histogram, so it suits quick comparisons, but it is not calibrated to any decoder.

In [8]:
expected_fault_weight = estimate_downstream_fault_weight(noisy_circuit, code=code)
model = ConsumerProfile.poisson(expected_fault_weight, consumer.correctable_weight)
model_phi, _ = histograms["with post-selection"].get_excess_failure_probability(model)
window_fault_weight = sum(calibration.mechanism_probabilities)
print(f"expected fault weight of the reference consumer: nu = {expected_fault_weight:.3f}")
print(f"expected fault weight of the window: {window_fault_weight:.3f}")
print(f"uncalibrated model: Phi = {model_phi:.2e} (calibrated: {phi:.2e})")

expected fault weight of the reference consumer: nu = 0.028
expected fault weight of the window: 0.068
uncalibrated model: Phi = 1.83e-04 (calibrated: 1.92e-04)


The uncalibrated model gives about 5% less than the calibrated $\Phi$.
Two effects pull in opposite directions.
The model's hard threshold counts every residual error of weight one together with a consumer fault of weight one as a failure, whereas the window's decoder corrects many such combinations.
The model's reference consumer also has fewer faults than the window: its $\nu$ is less than half of the window's expected fault weight.
Here the two effects nearly cancel, but this agreement need not hold for other circuits, and the model is not a bound.

## Next steps

The calibration above scores any circuit that prepares the same ideal block, with any noise and post-selection.
A new calibration is needed when the prepared state, the window's noise, or its decoder changes.
A preparation scheme that also supplies fresh $|+_L\rangle$ blocks is scored by a second calibration, with `basis="X"`, and the two scores add.
The [consumer-calibration guide](https://github.com/qLDPCOrg/qLDPC/blob/main/src/qldpc/experimental/circuits/benchmarking/consumer_calibration.md) explains the approximations of $\Phi$, custom consumers and decoders, rescaling, saving calibrations, and validation.
The [fault-tolerance certificates notebook](../logical_error_rates/5_fault_tolerance_certificates.ipynb) checks the separate yes-or-no criterion of bounded fault propagation.
The [API reference](https://qldpc.readthedocs.io/autoapi/qldpc/experimental/circuits/benchmarking/index.html) lists every argument.